# Generating bar plots with subplots for each features that were in atleast 2 classes

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# Create the directory to save the plots if it does not exist
output_folder = "dr_neal_experiments"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Load the data, telling pandas not to interpret "None" as NaN
df = pd.read_csv('Features_in_2__Conditions.csv', keep_default_na=False)

# Filter out the specified features
features_to_skip = ['C_ratio', 'type_token_ratio']
df_filtered = df[~df['Feature'].isin(features_to_skip)]

# Reshape the filtered data for plotting
df_melted = df_filtered.melt(
    id_vars=['Feature', 'Condition'],
    value_vars=['Mean (Correct)', 'Mean (Incorrect)'],
    var_name='Classification',
    value_name='Mean Value'
)

# Get unique features from the filtered data
features = df_melted['Feature'].unique()
n_features = len(features)

# Determine the grid size for subplots
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

# Create subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(10 * n_cols, 8 * n_rows))
axes = axes.flatten()

# To capture bar handles for the legend
bars1, bars2 = None, None

# Plot each feature in a separate subplot
for i, feature in enumerate(features):
    ax = axes[i]
    feature_data = df_melted[df_melted['Feature'] == feature]

    # Create a bar plot
    width = 0.35
    x = np.arange(len(feature_data['Condition'].unique()))

    correct_data = feature_data[feature_data['Classification'] == 'Mean (Correct)']
    incorrect_data = feature_data[feature_data['Classification'] == 'Mean (Incorrect)']

    # Plot the bars (and save the first set for the legend)
    b1 = ax.bar(x - width/2, correct_data['Mean Value'], width, label="Correct")
    b2 = ax.bar(x + width/2, incorrect_data['Mean Value'], width, label="Incorrect")

    if bars1 is None and bars2 is None:
        bars1, bars2 = b1, b2  # keep the first handles for the universal legend

    # Add labels and title with the requested font sizes
    ax.set_ylabel('Mean Value', fontsize=30)
    ax.set_title(feature, fontsize=24)
    ax.set_xticks(x)
    ax.set_xticklabels(correct_data['Condition'], fontsize=24)
    ax.tick_params(axis='y', labelsize=24)

# Hide empty subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

# Add the single, universal legend using the actual bar handles
fig.legend([bars1[0], bars2[0]], ["Correct", "Incorrect"],
           loc="upper center", bbox_to_anchor=(0.5, 1.02),
           ncol=2, fancybox=True, shadow=True, fontsize=20)

# Adjust layout to leave space for the legend
plt.tight_layout(rect=[0, 0, 1, 0.95])

# Save the plots in PNG and PDF formats with high resolution
plt.savefig(os.path.join(output_folder, 'features_subplot.png'),
            dpi=600, bbox_inches='tight')
plt.savefig(os.path.join(output_folder, 'features_subplot.pdf'),
            dpi=600, bbox_inches='tight')

# Close the plot to free up memory
plt.close()

print(f"Plots saved successfully in '{output_folder}' folder.")

Plots saved successfully in 'dr_neal_experiments' folder.


# Generating bar plots with subplots for each features that were in atleast 2 classes with error bars


In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# Create the directory to save the plots if it does not exist
output_folder = "dr_neal_experiments"
os.makedirs(output_folder, exist_ok=True)

# Load the data, telling pandas not to interpret "None" as NaN
df = pd.read_csv('Features_in_2__Conditions.csv', keep_default_na=False)

# Filter out the specified features (make a copy to avoid SettingWithCopyWarning)
features_to_skip = ['C_ratio', 'type_token_ratio']
df_filtered = df[~df['Feature'].isin(features_to_skip)].copy()

# Ensure numeric types for means and stds (in case CSV has strings)
num_cols = ['Mean (Correct)', 'Mean (Incorrect)', 'STD (Correct)', 'STD (Incorrect)']
for col in num_cols:
    df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce')

# Get unique features
features = df_filtered['Feature'].unique()
n_features = len(features)

# Determine the grid size for subplots
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

# Create subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(10 * n_cols, 8 * n_rows))
axes = np.atleast_1d(axes).flatten()

bars1, bars2 = None, None  # to capture legend handles

# Plot each feature in a separate subplot
for i, feature in enumerate(features):
    ax = axes[i]
    feature_data = df_filtered[df_filtered['Feature'] == feature]

    # X-axis positions by condition (preserve CSV order)
    conditions = feature_data['Condition'].values
    x = np.arange(len(conditions))
    width = 0.35

    # Data arrays
    means_correct = feature_data['Mean (Correct)'].to_numpy()
    means_incorrect = feature_data['Mean (Incorrect)'].to_numpy()
    std_correct = feature_data['STD (Correct)'].to_numpy()
    std_incorrect = feature_data['STD (Incorrect)'].to_numpy()

    # Build asymmetric yerr so lower side never goes below 0
    lower_correct = np.minimum(std_correct, means_correct)         # cap at mean
    lower_incorrect = np.minimum(std_incorrect, means_incorrect)   # cap at mean

    yerr_correct = np.vstack([lower_correct, std_correct])         # shape (2, N): [lower; upper]
    yerr_incorrect = np.vstack([lower_incorrect, std_incorrect])

    # Bars with error bars
    b1 = ax.bar(x - width/2, means_correct, width,
                yerr=yerr_correct, capsize=5, label="Correct")
    b2 = ax.bar(x + width/2, means_incorrect, width,
                yerr=yerr_incorrect, capsize=5, label="Incorrect")

    # Keep the first handles for the universal legend (ensures matching colors)
    if bars1 is None and bars2 is None:
        bars1, bars2 = b1, b2

    # Labels and formatting
    ax.set_ylabel('Mean Value', fontsize=30)
    ax.set_title(feature, fontsize=30)
    ax.set_xticks(x)
    ax.set_xticklabels(conditions, fontsize=24)
    ax.tick_params(axis='y', labelsize=24)
    ax.set_ylim(bottom=0)  # keep everything non-negative visually

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

# Universal legend using actual bar handles (colors match)
fig.legend([bars1[0], bars2[0]], ["Correct", "Incorrect"],
           loc="upper center", bbox_to_anchor=(0.5, 1.02),
           ncol=2, fancybox=True, shadow=True, fontsize=20)

# Layout & save
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(os.path.join(output_folder, 'features_subplot.png'), dpi=600, bbox_inches='tight')
plt.savefig(os.path.join(output_folder, 'features_subplot.pdf'), dpi=600, bbox_inches='tight')
plt.close()

print(f"Plots saved successfully in '{output_folder}' folder.")

Plots saved successfully in 'dr_neal_experiments' folder.
